# Training: Russian to Tatar Translator
Run `preprocess.ipynb` first if you haven't yet.

## 1. Hyperparameters

In [10]:
MODEL_VERSION = 'v2'             # Новая версия архитектуры
MAX_EPOCHS_PER_SESSION = 2      # Чуть больше за сессию
TOTAL_EPOCHS_TARGET = 40        # Больше эпох для выхода к loss < 1
BATCH_SIZE = 24
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
BETAS = (0.9, 0.98)
D_MODEL = 512
NUM_LAYERS = 6
DIM_FEEDFORWARD = 2048
DROPOUT = 0.2                   # Увеличена регуляризация
ACCUMULATION_STEPS = 8          # effective batch = 192
GRAD_CLIP_NORM = 1.0
LABEL_SMOOTHING = 0.15          # Чуть сильнее
MAX_LEN = 96                    # Должен совпадать с preprocess
USE_GRADIENT_CHECKPOINTING = True
USE_ROPE = True                 # Rotary Position Embedding


## 2. Imports & Device

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import sentencepiece as spm
import math
import os
import glob
import json
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print(f'Using device: {device}, AMP: {use_amp}')

Using device: cuda, AMP: True


## 3. Load Data & Tokenizer

In [12]:
train_data = torch.load('preprocessed/train_tokenized.pt', weights_only=False)
test_data = torch.load('preprocessed/test_tokenized.pt', weights_only=False)
sp = spm.SentencePieceProcessor(model_file='tokenizer/spm.model')
vocab_size = sp.get_piece_size()
print(f'Vocab size: {vocab_size}, Train samples: {len(train_data)}, Test samples: {len(test_data)}')

Vocab size: 22000, Train samples: 497500, Test samples: 2500


In [13]:
def collate_fn(batch):
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=sp.pad_id())
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=sp.pad_id())
    return {'src': src_padded, 'tgt': tgt_padded}

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

Train batches: 20730, Test batches: 105


## 4. Model Definition

In [14]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerMT(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=8, num_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(vocab_size, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward, dropout=dropout,
            norm_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_key_padding_mask=None):
        src_emb = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        output = self.transformer(
            src_emb.transpose(0, 1), tgt_emb.transpose(0, 1),
            src_mask=None, tgt_mask=tgt_mask, memory_mask=None,
            src_key_padding_mask=src_key_padding_mask, tgt_key_padding_mask=None,
        )
        return self.fc_out(output.transpose(0, 1))


def create_masks(src, tgt, pad_id):
    src_key_padding_mask = (src == pad_id)
    tgt_len = tgt.size(1)
    subsequent_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=tgt.device)
    return src_key_padding_mask, subsequent_mask

## 5. Load or Create Checkpoint

In [15]:
def find_latest_checkpoint(version):
    ckpt_dir = f'checkpoints/{version}'
    meta_path = f'{ckpt_dir}/meta.json'
    if not os.path.exists(meta_path):
        return ckpt_dir, 0, 0
    with open(meta_path) as f:
        meta = json.load(f)
    return ckpt_dir, meta['epoch'], meta.get('global_step', 0)


def save_checkpoint(ckpt_dir, model, optimizer, scheduler, scaler, epoch, global_step):
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save(model.state_dict(), f'{ckpt_dir}/model_epoch_{epoch}.pt')
    torch.save(optimizer.state_dict(), f'{ckpt_dir}/optimizer_epoch_{epoch}.pt')
    torch.save(scheduler.state_dict(), f'{ckpt_dir}/scheduler_epoch_{epoch}.pt')
    torch.save(scaler.state_dict(), f'{ckpt_dir}/scaler_epoch_{epoch}.pt')
    meta = {
        'epoch': epoch,
        'global_step': global_step,
        'hyperparameters': {
            'd_model': D_MODEL, 'num_layers': NUM_LAYERS,
            'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
            'dim_feedforward': DIM_FEEDFORWARD, 'dropout': DROPOUT,
            'accumulation_steps': ACCUMULATION_STEPS,
        }
    }
    with open(f'{ckpt_dir}/meta.json', 'w') as f:
        json.dump(meta, f, indent=2)


# --- Init model, optimizer, criterion, scaler ---
model = TransformerMT(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=BETAS)
criterion = nn.CrossEntropyLoss(ignore_index=sp.pad_id(), label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler('cuda' if use_amp else 'cpu')

# --- Load checkpoint if exists ---
ckpt_dir, start_epoch, global_step = find_latest_checkpoint(MODEL_VERSION)

if start_epoch > 0:
    print(f'Resuming from {ckpt_dir}, epoch {start_epoch}, global_step {global_step}')
    model.load_state_dict(torch.load(f'{ckpt_dir}/model_epoch_{start_epoch}.pt', weights_only=True))
    optimizer.load_state_dict(torch.load(f'{ckpt_dir}/optimizer_epoch_{start_epoch}.pt', weights_only=True))
    scaler.load_state_dict(torch.load(f'{ckpt_dir}/scaler_epoch_{start_epoch}.pt', weights_only=True))
else:
    os.makedirs(ckpt_dir, exist_ok=True)
    start_epoch = 0
    global_step = 0
    print(f'Starting fresh training -> {ckpt_dir}')

    # --- Scheduler (recalculated each session based on remaining epochs) ---
    remaining_epochs = min(MAX_EPOCHS_PER_SESSION, TOTAL_EPOCHS_TARGET - start_epoch)
    total_steps = (len(train_loader) * remaining_epochs) // ACCUMULATION_STEPS
    warmup_steps = max(1, int(0.1 * total_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps) *
            (0.1 + 0.9 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps))) / 2)
    )

    # Fast-forward scheduler to resume LR correctly
    for _ in range(global_step):
        scheduler.step()

print(f'Will train epochs {start_epoch+1} -> {start_epoch + remaining_epochs} (global_step={global_step})')

/home/rmssss/anaconda3/lib/python3.13/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Resuming from checkpoints/v2, epoch 4, global_step 10364
Will train epochs 5 -> 8 (global_step=10364)


## 6. Training Loop

In [ ]:
end_epoch = start_epoch + remaining_epochs

for epoch in range(start_epoch, end_epoch):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{end_epoch}')

    for step, batch in enumerate(pbar):
        src = batch['src'].to(device)
        tgt = batch['tgt'].to(device)
        src_key_padding_mask, subsequent_mask = create_masks(src, tgt, sp.pad_id())

        with torch.amp.autocast('cuda' if use_amp else 'cpu'):
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            tgt_input_len = tgt_input.size(1)
            output = model(
                src, tgt_input,
                src_key_padding_mask=src_key_padding_mask,
                tgt_mask=subsequent_mask[:tgt_input_len, :tgt_input_len]
            )
            loss = criterion(output.reshape(-1, vocab_size), tgt_output.reshape(-1))
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scale_before <= scaler.get_scale():
                scheduler.step()
                global_step += 1
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix({
            'loss': f'{total_loss / (step + 1):.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })

    # Save checkpoint after each epoch
    save_checkpoint(ckpt_dir, model, optimizer, scheduler, scaler, epoch + 1, global_step)
    print(f'Epoch {epoch+1} finished, Avg Loss: {total_loss/len(train_loader):.4f}')

print(f'Done. Saved to {ckpt_dir}')

Epoch 5/8: 100%|██████████████████████████████████████| 20730/20730 [1:40:01<00:00,  3.45it/s, loss=5.0981, lr=7.82e-05]


Epoch 5 finished, Avg Loss: 5.0981


Epoch 6/8:   1%|▎                                       | 131/20730 [00:45<1:34:08,  3.65it/s, loss=5.0566, lr=7.87e-05]

## 7. Test Translations

In [ ]:
def translate(model, sp, sentence, max_len=128, repetition_penalty=1.2, temperature=1.0):
    model.eval()
    src_tokens = sp.encode(sentence)[:max_len-2]
    src = torch.tensor([[sp.bos_id()] + src_tokens + [sp.eos_id()]], dtype=torch.long, device=device)
    src_key_padding_mask = (src == sp.pad_id())

    with torch.no_grad():
        src_emb = model.pos_enc(model.src_embed(src) * math.sqrt(model.d_model)).transpose(0, 1)
        memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        tgt_in = torch.tensor([[sp.bos_id()]], dtype=torch.long, device=device)

        for _ in range(max_len):
            tgt_emb = model.pos_enc(model.tgt_embed(tgt_in) * math.sqrt(model.d_model)).transpose(0, 1)
            tgt_len = tgt_in.size(1)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=device)
            out = model.transformer.decoder(
                tgt_emb, memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_key_padding_mask
            )
            logits = model.fc_out(out.transpose(0, 1))[:, -1, :] / temperature
            for token in set(tgt_in[0].tolist()):
                if token not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]:
                    if logits[0, token] > 0:
                        logits[0, token] /= repetition_penalty
                    else:
                        logits[0, token] *= repetition_penalty
            next_word = logits.argmax(-1).item()
            if next_word == sp.eos_id():
                break
            tgt_in = torch.cat((tgt_in, torch.tensor([[next_word]], device=device)), dim=1)

    return sp.decode(tgt_in[0, 1:].tolist())

In [ ]:
test_sentences = [
    'Привет мир',
    'Я тебя люблю',
    'Пойдем завтра в кино',
    'Собаки и кошки',
    'Тебе нужно посадить дерево, вырастить ребенка и построить дом',
    'Уроки фехтования',
    'Ретроградный меркурий сегодня сильно повлиял на меня',
]

for s in test_sentences:
    print(f"'{s}' -> {translate(model, sp, s)}")